# 筛选apparent temperature > 33 ℃的数据

In [6]:
import glob
import os
import pandas as pd
input_dir = r'D:\seoul\a_airtem'
csv_file = os.path.join(input_dir, 'AT_2015_2024.csv')
df = pd.read_csv(csv_file, skiprows=3, encoding='euc_kr')

df

,일자,기온(°C),습도(%rh),체감온도(°C)
0,2015-05-01,27.0,46.6,26.8
1,2015-05-02,26.0,40.4,25.3
2,2015-05-03,19.1,95.9,22.5
3,2015-05-04,17.1,90.0,20.1
4,2015-05-05,19.1,26.3,17.6
...,...,...,...,...
1525,2024-09-26,28.0,57.9,28.3
1526,2024-09-27,27.7,57.1,27.9
1527,2024-09-28,27.9,51.7,27.6
1528,2024-09-29,28.9,49.0,28.4


In [62]:
print(df)

                Landsat Product Identifier L2  \
0    LC08_L2SP_116034_20150501_20200909_02_T1   
1    LC08_L2SP_115034_20150510_20200909_02_T1   
2    LC08_L2SP_115035_20150510_20200909_02_T1   
3    LC08_L2SP_116034_20150517_20200909_02_T1   
4    LC08_L2SP_115034_20150526_20200909_02_T1   
..                                        ...   
356  LC09_L2SP_115035_20240915_20240916_02_T1   
357  LC09_L2SP_116034_20240922_20240924_02_T1   
358  LC08_L2SP_115034_20240923_20240928_02_T1   
359  LC08_L2SP_115035_20240923_20240928_02_T1   
360  LC08_L2SP_116034_20240930_20241005_02_T1   

                Landsat Product Identifier L1 Landsat Scene Identifier  \
0    LC08_L1TP_116034_20150501_20200909_02_T1    LC81160342015121LGN01   
1    LC08_L1TP_115034_20150510_20200909_02_T1    LC81150342015130LGN01   
2    LC08_L1TP_115035_20150510_20200909_02_T1    LC81150352015130LGN01   
3    LC08_L1TP_116034_20150517_20200909_02_T1    LC81160342015137LGN01   
4    LC08_L1TP_115034_20150526_20200909_0

In [8]:
import pandas as pd

# 1. 读取并解析日期
path = r'D:\seoul\a_airtem\AT_2015_2024.csv'
df = pd.read_csv(path, skiprows=3, encoding='euc_kr', parse_dates=['일자'])

# 2. 按日期排序
df = df.sort_values('일자')
print(df)
# 3. 标记“体感温度 > 33”
df['is_hot'] = df['체감온도(°C)'] > 33

# 4. 计算前一天记录与当前行的日期差
df['prev_diff'] = df['일자'] - df['일자'].shift(1)

# 5. 标记“与前一天连续且两天都 >33”
df['two_day_consec'] = (
    df['is_hot'] &
    df['is_hot'].shift(1) &
    (df['prev_diff'] == pd.Timedelta(days=1))
)

# 6. 把“连续两天”中的两个日期都选出来
mask = df['two_day_consec'] | df['two_day_consec'].shift(-1).fillna(False)
result = df.loc[mask, ['일자', '기온(°C)', '습도(%rh)', '체감온도(°C)']]

# 7. 输出
print("=== 连续两天体感温度 > 33°C 的记录 ===")
print(result.to_string(index=False))
result.to_csv(r'D:\seoul\a_airtem\result\highest_temps_over_33.csv')

             일자  기온(°C)  습도(%rh)  체감온도(°C)
0    2015-05-01    27.0     46.6      26.8
1    2015-05-02    26.0     40.4      25.3
2    2015-05-03    19.1     95.9      22.5
3    2015-05-04    17.1     90.0      20.1
4    2015-05-05    19.1     26.3      17.6
...         ...     ...      ...       ...
1525 2024-09-26    28.0     57.9      28.3
1526 2024-09-27    27.7     57.1      27.9
1527 2024-09-28    27.9     51.7      27.6
1528 2024-09-29    28.9     49.0      28.4
1529 2024-09-30    28.4     46.8      27.7

[1530 rows x 4 columns]
=== 连续两天体感温度 > 33°C 的记录 ===
        일자  기온(°C)  습도(%rh)  체감온도(°C)
2015-07-10    33.5     48.8      33.4
2015-07-11    35.9     43.0      35.1
2015-07-30    32.9     59.4      33.8
2015-07-31    32.3     62.0      33.4
2015-08-06    34.3     50.2      34.3
2015-08-07    34.4     50.9      34.5
2015-08-08    32.3     60.1      33.3
2016-07-30    32.8     54.7      33.3
2016-07-31    31.8     64.2      33.1
2016-08-01    32.0     61.6      33.1
2016-08-03   

C:\Users\owner\AppData\Local\Temp\ipykernel_51388\3979580143.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = df['two_day_consec'] | df['two_day_consec'].shift(-1).fillna(False)


In [9]:
import pandas as pd

path = r'D:\seoul\a_airtem\result\highest_temps_over_33.csv'

# 用和写出时一样的编码来读取
df = pd.read_csv(path, encoding='utf-8-sig', parse_dates=['일자'])

# 现在就能打印 일자 列了
print(df['일자'])

0     2015-07-10
1     2015-07-11
2     2015-07-30
3     2015-07-31
4     2015-08-06
         ...    
158   2024-09-10
159   2024-09-11
160   2024-09-17
161   2024-09-18
162   2024-09-19
Name: 일자, Length: 163, dtype: datetime64[ns]


In [10]:
import numpy as np

unique_dates = np.array(df['일자'].dt.strftime('%Y-%m-%d'))
print(unique_dates)

['2015-07-10' '2015-07-11' '2015-07-30' '2015-07-31' '2015-08-06'
 '2015-08-07' '2015-08-08' '2016-07-30' '2016-07-31' '2016-08-01'
 '2016-08-03' '2016-08-04' '2016-08-05' '2016-08-06' '2016-08-07'
 '2016-08-08' '2016-08-09' '2016-08-10' '2016-08-11' '2016-08-12'
 '2016-08-13' '2016-08-14' '2016-08-15' '2016-08-16' '2016-08-17'
 '2016-08-18' '2016-08-19' '2016-08-20' '2016-08-21' '2016-08-22'
 '2016-08-23' '2017-07-19' '2017-07-20' '2017-07-21' '2017-07-22'
 '2017-08-01' '2017-08-02' '2017-08-03' '2017-08-04' '2017-08-05'
 '2017-08-06' '2017-08-07' '2018-07-15' '2018-07-16' '2018-07-19'
 '2018-07-20' '2018-07-21' '2018-07-22' '2018-07-23' '2018-07-24'
 '2018-07-25' '2018-07-26' '2018-07-27' '2018-07-28' '2018-07-29'
 '2018-07-30' '2018-07-31' '2018-08-01' '2018-08-02' '2018-08-03'
 '2018-08-04' '2018-08-05' '2018-08-06' '2018-08-07' '2018-08-08'
 '2018-08-09' '2018-08-10' '2018-08-11' '2018-08-12' '2018-08-13'
 '2018-08-14' '2018-08-15' '2018-08-16' '2019-08-02' '2019-08-03'
 '2019-08-

# 02 需要下载全部的对应文件的 B10 和 Qixel 处理 extreme heat date
 https://earthexplorer.usgs.gov/

In [11]:
landsat_8 = r'D:\seoul\b_satellite_img\landsat_ot_c2_l2.csv'
df = pd.read_csv(landsat_8, encoding='ISO-8859-1')
df

,Landsat Product Identifier L2,Landsat Product Identifier L1,Landsat Scene Identifier,Date Acquired,Collection Category,Collection Number,WRS Path,WRS Row,Target WRS Path,Target WRS Row,...,Corner Upper Left Latitude,Corner Upper Left Longitude,Corner Upper Right Latitude,Corner Upper Right Longitude,Corner Lower Left Latitude,Corner Lower Left Longitude,Corner Lower Right Latitude,Corner Lower Right Longitude,Display ID,Entity ID
0,LC08_L2SP_116034_20150501_20200909_02_T1,LC08_L1TP_116034_20150501_20200909_02_T1,LC81160342015121LGN01,2015/05/01,T1,2,116,34,116,34,...,38.51088,125.21327,38.56704,127.90608,36.36027,125.32025,36.41226,127.93707,LC08_L2SP_116034_20150501_20200909_02_T1,LC81160342015121LGN01
1,LC08_L2SP_115034_20150510_20200909_02_T1,LC08_L1TP_115034_20150510_20200909_02_T1,LC81150342015130LGN01,2015/05/10,T1,2,115,34,115,34,...,38.51915,126.80568,38.53876,129.47274,36.38712,126.86713,36.40528,129.45949,LC08_L2SP_115034_20150510_20200909_02_T1,LC81150342015130LGN01
2,LC08_L2SP_115035_20150510_20200909_02_T1,LC08_L1TP_115035_20150510_20200909_02_T1,LC81150352015130LGN01,2015/05/10,T1,2,115,35,115,35,...,37.08357,126.40232,37.11203,129.02139,34.95164,126.47141,34.97797,129.02082,LC08_L2SP_115035_20150510_20200909_02_T1,LC81150352015130LGN01
3,LC08_L2SP_116034_20150517_20200909_02_T1,LC08_L1TP_116034_20150517_20200909_02_T1,LC81160342015137LGN01,2015/05/17,T1,2,116,34,116,34,...,38.51253,125.26477,38.56755,127.96117,36.36180,125.37031,36.41272,127.99060,LC08_L2SP_116034_20150517_20200909_02_T1,LC81160342015137LGN01
4,LC08_L2SP_115034_20150526_20200909_02_T1,LC08_L1TP_115034_20150526_20200909_02_T1,LC81150342015146LGN01,2015/05/26,T1,2,115,34,115,34,...,38.52010,126.85726,38.53852,129.52782,36.38801,126.91727,36.40507,129.51302,LC08_L2SP_115034_20150526_20200909_02_T1,LC81150342015146LGN01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,LC09_L2SP_115035_20240915_20240916_02_T1,LC09_L1TP_115035_20240915_20240915_02_T1,LC91150352024259LGN00,2024/09/15,T1,2,115,35,115,35,...,37.08408,126.42592,37.11202,129.04840,34.95212,126.49438,34.97796,129.04711,LC09_L2SP_115035_20240915_20240916_02_T1,LC91150352024259LGN00
357,LC09_L2SP_116034_20240922_20240924_02_T1,LC09_L1TP_116034_20240922_20240922_02_T1,LC91160342024266LGN00,2024/09/22,T1,2,116,34,116,34,...,38.51016,125.27521,38.56493,127.97154,36.36211,125.38032,36.41280,128.00063,LC09_L2SP_116034_20240922_20240924_02_T1,LC91160342024266LGN00
358,LC08_L2SP_115034_20240923_20240928_02_T1,LC08_L1TP_115034_20240923_20240928_02_T1,LC81150342024267LGN00,2024/09/23,T1,2,115,34,115,34,...,38.51979,126.84007,38.53860,129.51060,36.38771,126.90056,36.40514,129.49629,LC08_L2SP_115034_20240923_20240928_02_T1,LC81150342024267LGN00
359,LC08_L2SP_115035_20240923_20240928_02_T1,LC08_L1TP_115035_20240923_20240928_02_T1,LC81150352024267LGN00,2024/09/23,T1,2,115,35,115,35,...,37.08437,126.43941,37.11201,129.05853,34.95239,126.50751,34.97796,129.05697,LC08_L2SP_115035_20240923_20240928_02_T1,LC81150352024267LGN00


In [12]:
# 过滤只保留 Collection Category 为 T1 的数据
df_t1 = df[df['Collection Category'] == 'T1']
df_t1['Date Acquired']

0      2015/05/01
1      2015/05/10
2      2015/05/10
3      2015/05/17
4      2015/05/26
          ...    
356    2024/09/15
357    2024/09/22
358    2024/09/23
359    2024/09/23
360    2024/09/30
Name: Date Acquired, Length: 298, dtype: object

In [13]:
unique_dates = unique_dates
acquired    = df_t1['Date Acquired'].to_numpy()
#print(unique_dates)
#print(acquired)
# 打印类型和元素的数据类型（如果有 dtype 属性）
print("unique_dates →", type(unique_dates), getattr(unique_dates, 'dtype', None))
print("acquired    →", type(acquired), acquired.dtype)

unique_dates → <class 'numpy.ndarray'> object
acquired    → <class 'numpy.ndarray'> object


In [14]:
import pandas as pd

unique_dates = unique_dates
acquired    = pd.to_datetime(
    df_t1['Date Acquired'],
    format='%Y/%m/%d',  # 注意这里是“/”
    errors='coerce'
).values
unique_dates_clean = unique_dates.astype('datetime64[D]')
acquired_clean      = acquired.astype   ('datetime64[D]')

In [15]:
import pandas as pd

# 求交集
common = set(acquired_clean) & set(unique_dates_clean)
# 把 set 转成列表并排序
common_sorted = sorted(common)
# （如果需要字符串格式，可以再 map 一次 str）
common_str = [str(d) for d in common_sorted]
print(f"共有 {len(common_str)} 个相同的日期：{common_str}")
for d in common_str:
    print(d)

共有 21 个相同的日期：['2016-07-31', '2016-08-07', '2016-08-16', '2016-08-23', '2017-08-03', '2018-07-21', '2018-07-28', '2018-08-06', '2018-08-13', '2019-08-09', '2020-08-18', '2021-07-29', '2021-08-05', '2023-07-27', '2023-08-03', '2023-08-04', '2023-08-19', '2024-08-05', '2024-08-06', '2024-08-13', '2024-08-14']
2016-07-31
2016-08-07
2016-08-16
2016-08-23
2017-08-03
2018-07-21
2018-07-28
2018-08-06
2018-08-13
2019-08-09
2020-08-18
2021-07-29
2021-08-05
2023-07-27
2023-08-03
2023-08-04
2023-08-19
2024-08-05
2024-08-06
2024-08-13
2024-08-14


In [16]:
sorted_dates = [str(d).replace("-", "") for d in common_sorted]
print(sorted_dates)  # 输出: 20240618

['20160731', '20160807', '20160816', '20160823', '20170803', '20180721', '20180728', '20180806', '20180813', '20190809', '20200818', '20210729', '20210805', '20230727', '20230803', '20230804', '20230819', '20240805', '20240806', '20240813', '20240814']


# 然后下载对应的文件

In [17]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    # Ensure the output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    nyc_shape = gpd.read_file(shapefile_path)

    # Loop through each raster in the raster folder
    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Ensure the shapefile is in the same CRS as the raster
            if nyc_shape.crs != raster_crs:
                nyc_shape = nyc_shape.to_crs(raster_crs)

            # Clip the raster using the shapefile
            out_image, out_transform = mask(src, nyc_shape.geometry, crop=True)

            # Update metadata for the output raster
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs  # Use the raster's CRS for the output
            })

            # Save the clipped raster to the output folder
            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            if os.path.isdir(output_raster_path):
                continue
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"Clipped raster saved to: {output_raster_path}")

In [18]:
# Paths to the folder and shapefile
raster_folder = r'D:\seoul\b_satellite_img\extreme\original'
seoul_boundary_file = r'D:\seoul\Final_data\Admin_boundary\Seoul_boundary.shp'
shp_seoul = seoul_boundary_file
output_folder = r'D:\seoul\b_satellite_img\extreme\clipped'
os.makedirs(output_folder, exist_ok=True)

clip_rasters_with_shapefile(raster_folder, shp_seoul, output_folder)

Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_SR_B4.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_SR_B5.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_ST_B10.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160823_20200906_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160823_20200906_02_T1_SR_B4.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160823_20200906_02_T1_SR_B5.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160823_20200906_02_T1_ST_

In [20]:
import os
import rasterio
import numpy as np
import pandas as pd
import glob
# Mapping from month names to numbers for date comparison
month_to_num = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
    'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for sorted_data in sorted_dates:
        pattern = r'{}\*_*{}_*_QA_PIXEL.TIF'.format(base_folder, sorted_data)
        matched_files = glob.glob(pattern)
        if not matched_files:
            continue
        with rasterio.open(matched_files[0]) as src:
            qa_pixel_data = src.read(1)
        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size
        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))
        pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

        print(f"📅 {sorted_data}:")
        val_all = 0
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
            val_all = val_all + prop
        print(val_all)


# Example usage
base_folder = r'D:\seoul\b_satellite_img\extreme\clipped'

calculate_pixel_proportion(sorted_dates, base_folder)

📅 20160807:
  - 像素值 21824 的占比: 78.39%
  - 像素值 21952 的占比: 4.13%
0.8252899697584389
📅 20160823:
  - 像素值 21824 的占比: 19.05%
  - 像素值 21952 的占比: 0.12%
0.1916066665180595
📅 20180813:
  - 像素值 21824 的占比: 1.81%
  - 像素值 21952 的占比: 0.00%
0.018127103720380733
📅 20200818:
  - 像素值 21824 的占比: 11.17%
  - 像素值 21952 的占比: 0.24%
0.1141184250611147
📅 20210805:
  - 像素值 21824 的占比: 14.41%
  - 像素值 21952 的占比: 0.65%
0.15064606971162778
📅 20230803:
  - 像素值 21824 的占比: 30.11%
  - 像素值 21952 的占比: 0.59%
0.3069243515154217
📅 20230804:
  - 像素值 21824 的占比: 14.74%
  - 像素值 21952 的占比: 0.80%
0.1553925830156855
📅 20230819:
  - 像素值 21824 的占比: 68.86%
  - 像素值 21952 的占比: 3.80%
0.7265583320330204
📅 20240805:
  - 像素值 21824 的占比: 8.68%
  - 像素值 21952 的占比: 0.25%
0.08935749686067333
📅 20240806:
  - 像素值 21824 的占比: 8.68%
  - 像素值 21952 的占比: 0.25%
0.08935749686067333
📅 20240813:
  - 像素值 21824 的占比: 59.70%
  - 像素值 21952 的占比: 3.25%
0.6295118997198754


# extreme heat day:
finally we choose the 20230819(0.726) and 20160807(0.825)
# normal heat day:
then I need to check the normal heat day percentile (40-60%) during the recent 30 years （1995-2024）

In [23]:
import os
import pandas as pd

input_dir = r'D:\seoul\a_airtem'
csv_files = [
    os.path.join(input_dir, 'AT_1995_2004.csv'),
    os.path.join(input_dir, 'AT_2005_2014.csv'),
    os.path.join(input_dir, 'AT_2015_2024.csv'),
]

# 读取并合并
dfs = [pd.read_csv(f, skiprows=3, encoding='euc_kr') for f in csv_files]
tem = pd.concat(dfs, ignore_index=True)

# 查看合并结果
print(tem.head())
tem.to_csv(r'D:\seoul\a_airtem\result\tem.csv')

           일자  기온(°C)  습도(%rh)  체감온도(°C)
0  1995-05-01    25.6     25.0      23.2
1  1995-05-02    22.0     40.0      21.4
2  1995-05-03    16.8     31.0      15.9
3  1995-05-04    17.2     23.0      15.6
4  1995-05-05    17.8     42.0      17.6


In [25]:
file = r'D:\seoul\a_airtem\result\tem.csv'
data = pd.read_csv(file)
data
# 计算每行 AT(체감온도(°C)) 的百分等级（0~100）
data['percentile'] = data['체감온도(°C)'].rank(pct=True) * 100

# 只保留你关心的几列
out_cols = ['일자', '기온(°C)', '습도(%rh)', '체감온도(°C)', 'percentile']
data_out = data[out_cols]

# 6. 保存为 UTF-8-SIG 编码的 CSV
out_fp = r'D:\seoul\a_airtem\result\temps.csv'
data_out.to_csv(out_fp, index=False, encoding='utf-8-sig')
print(f"✔ 已保存 {len(data_out)} 条 5/6/7/8/9 月记录（含 percentile 列）到：{out_fp}")


✔ 已保存 4590 条 5/6/7/8/9 月记录（含 percentile 列）到：D:\seoul\a_airtem\result\temps.csv


In [26]:
file = r'D:\seoul\a_airtem\result\temps.csv'
data = pd.read_csv(file)
data

,일자,기온(°C),습도(%rh),체감온도(°C),percentile
0,1995-05-01,25.6,25.0,23.2,13.823529
1,1995-05-02,22.0,40.0,21.4,7.287582
2,1995-05-03,16.8,31.0,15.9,0.152505
3,1995-05-04,17.2,23.0,15.6,0.108932
4,1995-05-05,17.8,42.0,17.6,0.784314
...,...,...,...,...,...
4585,2024-09-26,28.0,57.9,28.3,52.875817
4586,2024-09-27,27.7,57.1,27.9,49.226580
4587,2024-09-28,27.9,51.7,27.6,46.448802
4588,2024-09-29,28.9,49.0,28.4,53.954248


In [35]:
file = r'D:\seoul\a_airtem\result\temps.csv'
data = pd.read_csv(file)
data

# 条件筛选：百分位在 40 到 60 之间的行
filtered_data = data[(data['percentile'] >= 40) & (data['percentile'] <= 60)]

# 打印这些行的 '일시' 列
print(filtered_data[['일자', 'percentile']])


              일자  percentile
38    1995-06-08   48.409586
39    1995-06-09   49.226580
40    1995-06-10   53.954248
43    1995-06-13   55.163399
46    1995-06-16   44.542484
...          ...         ...
4585  2024-09-26   52.875817
4586  2024-09-27   49.226580
4587  2024-09-28   46.448802
4588  2024-09-29   53.954248
4589  2024-09-30   47.505447

[926 rows x 2 columns]


In [36]:
# 转换日期格式：去掉 "-"，筛选以 2024 或 2016 开头的日期
sorted_dates = [str(d).replace("-", "") for d in filtered_data['일자']]
filtered_dates = [d for d in sorted_dates if d.startswith('2023') or d.startswith('2016')]

# 用 boolean mask 获取对应的行（原始 '일시' 格式）
filtered_rows = filtered_data[filtered_data['일자'].astype(str).str.replace("-", "").isin(filtered_dates)]

# 打印日期和 percentile 两列
filtered_rows[['일자', 'percentile']]


,일자,percentile
3231,2016-05-19,55.163399
3234,2016-05-22,52.875817
3235,2016-05-23,45.413943
3240,2016-05-28,41.808279
3242,2016-05-30,44.542484
...,...,...
4406,2023-08-31,57.592593
4420,2023-09-14,49.226580
4422,2023-09-16,55.163399
4425,2023-09-19,52.026144


In [37]:
import numpy as np
import pandas as pd

# 将 '일시' 列转换为字符串形式的日期
normal_dates_clean = pd.to_datetime(filtered_data['일자']).dt.strftime('%Y-%m-%d')
filtered_dates = [d for d in normal_dates_clean if d.startswith('2023') or d.startswith('2016')]

set_normal_array = np.array(filtered_dates)

# 将 acquired_clean 转为字符串形式
acquired_str_array = np.array([str(d)[:10] for d in acquired_clean])  # 保留到日，防止有时间部分

# 取交集
common = np.intersect1d(acquired_str_array, set_normal_array)

# 排序并转成列表
common_sorted = sorted(common.tolist())
print(f"共有 {len(common_sorted)} 个相同的日期：{common_sorted}")


共有 11 个相同的日期：['2016-05-19', '2016-05-28', '2016-07-06', '2016-09-08', '2016-09-17', '2016-09-24', '2023-05-31', '2023-06-09', '2023-06-16', '2023-08-28', '2023-09-28']


In [42]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    # Ensure the output directory exists
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    nyc_shape = gpd.read_file(shapefile_path)

    # Loop through each raster in the raster folder
    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Ensure the shapefile is in the same CRS as the raster
            if nyc_shape.crs != raster_crs:
                nyc_shape = nyc_shape.to_crs(raster_crs)

            # Clip the raster using the shapefile
            out_image, out_transform = mask(src, nyc_shape.geometry, crop=True)

            # Update metadata for the output raster
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs  # Use the raster's CRS for the output
            })

            # Save the clipped raster to the output folder
            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            if os.path.isdir(output_raster_path):
                continue
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"Clipped raster saved to: {output_raster_path}")

In [43]:
# Paths to the folder and shapefile
raster_folder = r'D:\seoul\b_satellite_img\normal\original'
seoul_boundary_file = r'D:\seoul\Final_data\Admin_boundary\Seoul_boundary.shp'
shp_seoul = seoul_boundary_file
output_folder = r'D:\seoul\b_satellite_img\normal\clipped'
os.makedirs(output_folder, exist_ok=True)

clip_rasters_with_shapefile(raster_folder, shp_seoul, output_folder)

Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_115034_20160528_20200906_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_115034_20160528_20200906_02_T1_ST_B10.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160519_20200907_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160519_20200907_02_T1_ST_B10.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160706_20200906_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160706_20200906_02_T1_ST_B10.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160908_20200906_02_T1_QA_PIXEL.TIF
Clipped raster saved to: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160908_20200906_02_T1_ST_

In [45]:
import os
import rasterio
import numpy as np
import pandas as pd
import glob
# Mapping from month names to numbers for date comparison
month_to_num = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
    'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for sorted_data in sorted_dates:
        pattern = r'{}\*_*{}_*_QA_PIXEL.TIF'.format(base_folder, sorted_data)
        matched_files = glob.glob(pattern)
        if not matched_files:
            continue
        with rasterio.open(matched_files[0]) as src:
            qa_pixel_data = src.read(1)
        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size
        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))
        pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

        print(f"📅 {sorted_data}:")
        val_all = 0
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
            val_all = val_all + prop
        print(val_all)


# Example usage
base_folder = r'D:\seoul\b_satellite_img\normal\clipped'

calculate_pixel_proportion(sorted_dates, base_folder)

📅 20160519:
  - 像素值 21824 的占比: 95.42%
  - 像素值 21952 的占比: 4.50%
0.999201979447627
📅 20160528:
  - 像素值 21824 的占比: 7.65%
  - 像素值 21952 的占比: 0.02%
0.07674223001388311
📅 20160706:
  - 像素值 21824 的占比: 15.72%
  - 像素值 21952 的占比: 0.28%
0.15999494735590677
📅 20160908:
  - 像素值 21824 的占比: 22.72%
  - 像素值 21952 的占比: 0.75%
0.23469234598723465
📅 20160924:
  - 像素值 21824 的占比: 94.84%
  - 像素值 21952 的占比: 4.62%
0.9946189340407035
📅 20230531:
  - 像素值 21824 的占比: 83.65%
  - 像素值 21952 的占比: 2.50%
0.8614624432506334
📅 20230616:
  - 像素值 21824 的占比: 94.80%
  - 像素值 21952 的占比: 4.21%
0.9900418329209484
📅 20230928:
  - 像素值 21824 的占比: 27.93%
  - 像素值 21952 的占比: 1.85%
0.2977850099938328


# 确定了 extreme heat and normal heat days
extreme heat 20160807, 20230819
normal heat 20160924, 20230616

# 下一步是得到 heat resilience map

# 用 QA数据清理B10 -> clean data

In [63]:
import rasterio
import numpy as np
import os
import glob

extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
label_dict = {d: 'extreme' for d in extreme_heat}
label_dict.update({d: 'normal' for d in normal_heat})

for date, label in label_dict.items():
    print(f"{label} heat day:", date)
    data_workspace = rf"D:\seoul\b_satellite_img\{label}\clipped"
    b_10_data = glob.glob(os.path.join(data_workspace, rf'clipped_*_{date}_*_ST_B10.TIF'))
    qa_data = glob.glob(os.path.join(data_workspace, rf'clipped_*_{date}_*_QA_PIXEL.TIF'))
    output_folder = os.path.join(data_workspace, r'clean')
    os.makedirs(output_folder, exist_ok=True)

    if not b_10_data or not qa_data:
        print(f"No file found for date {date}")
        continue

    b_10_data = b_10_data[0]
    qa_data = qa_data[0]

    output_data = os.path.join(output_folder, rf'clipped_{date}_B10_cleaned.tif')
    print("B10 Data Path:", b_10_data)
    print("QA Data Path:", qa_data)
    print("Output Data Path:", output_data)

    # Read Band 10 and QA data
    with rasterio.open(b_10_data) as src:
        band10 = src.read(1).astype(float)
        profile = src.profile

    with rasterio.open(qa_data) as qa_src:
        qa = qa_src.read(1)

    # QA filtering
    valid_labels = [21824, 21952]
    cloud_mask = ~np.isin(qa, valid_labels)
    band10[cloud_mask] = 0

    with rasterio.open(output_data, 'w', **profile) as dst:
        dst.write(band10, 1)



extreme heat day: 20160807
B10 Data Path: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_ST_B10.TIF
QA Data Path: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC08_L2SP_116034_20160807_20200906_02_T1_QA_PIXEL.TIF
Output Data Path: D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_20160807_B10_cleaned.tif
extreme heat day: 20230819
B10 Data Path: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC09_L2SP_116034_20230819_20230821_02_T1_ST_B10.TIF
QA Data Path: D:\seoul\b_satellite_img\extreme\clipped\clipped_LC09_L2SP_116034_20230819_20230821_02_T1_QA_PIXEL.TIF
Output Data Path: D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_20230819_B10_cleaned.tif
normal heat day: 20160924
B10 Data Path: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160924_20200906_02_T1_ST_B10.TIF
QA Data Path: D:\seoul\b_satellite_img\normal\clipped\clipped_LC08_L2SP_116034_20160924_20200906_02_T1_QA_PIXEL.TIF
Output Data Path: D:\seoul\

In [3]:
import os
import rasterio
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape

def raster_to_shapefile(raster_files, output_folder):
    # 确保输出目录存在
    os.makedirs(output_folder, exist_ok=True)

    for raster_file in raster_files:
        with rasterio.open(raster_file) as src:
            # 获取栅格的nodata值
            nodata_value = src.nodata
            # 输出 Shapefile 文件路径
            output_shapefile = os.path.join(output_folder, os.path.basename(raster_file).replace('.tif', '.shp'))
            if os.path.exists(output_shapefile):
                continue
            # 保存为 Shapefile
            gdf.to_file(output_shapefile)

            # 提取栅格的形状
            mask = src.read(1)  # 读取第一个波段的数据
            results = shapes(mask, mask=mask != nodata_value, transform=src.transform)

            # 将结果转换为 GeoDataFrame
            shapes_list = []
            for geom, value in results:
                geom = shape(geom)
                if geom.is_valid:
                    shapes_list.append({"geometry": geom, "value": value})

            # 创建 GeoDataFrame
            gdf = gpd.GeoDataFrame(shapes_list, crs=src.crs)


            print(f"栅格已转换为 Shapefile 并保存到: {output_shapefile}")
extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files = [
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[1]}_B10_cleaned.tif'
]
extreme_files = [
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[1]}_B10_cleaned.tif'
]

# 合并所有栅格文件
all_raster_files = normal_files + extreme_files

# 输出文件夹路径
output_shapefile_folder = r'D:\seoul\b_satellite_img\shapefiles'
os.makedirs(output_shapefile_folder, exist_ok=True)

# 转换栅格文件为 Shapefile
raster_to_shapefile(all_raster_files, output_shapefile_folder)

# 得到fid的shape文件

In [4]:
import geopandas as gpd
import pandas as pd
import os
import glob

def fast_clip_and_assign_fid(input_shp, clip_shp, output_shp):
    print(f"📂 加载输入数据：{input_shp}")
    input_gdf = gpd.read_file(input_shp)
    clip_gdf = gpd.read_file(clip_shp)

    # 确保 CRS 一致
    if input_gdf.crs != clip_gdf.crs:
        input_gdf = input_gdf.to_crs(clip_gdf.crs)

    # 在裁剪前先加上 city_id 字段
    clip_gdf = clip_gdf.reset_index().rename(columns={'index': 'city_id'})

    print("✂️ 正在执行 overlay 精准裁剪...")
    try:
        clipped = gpd.overlay(input_gdf, clip_gdf[['city_id', 'geometry']], how='intersection', keep_geom_type=False)
    except Exception as e:
        print(f"❌ Overlay 错误: {e}")
        return

    # 保存结果
    os.makedirs(os.path.dirname(output_shp), exist_ok=True)
    clipped.to_file(output_shp, driver='ESRI Shapefile')
    print(f"✅ 已保存: {output_shp}")
extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]

# 输入的栅格转 Shapefile 路径（假设已经将其转化为 Shapefile）
shapefile_files = [
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{normal_heat[0]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{normal_heat[1]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{extreme_heat[0]}_B10_cleaned.shp',
    rf'D:\seoul\b_satellite_img\shapefiles\clipped_{extreme_heat[1]}_B10_cleaned.shp'

]
grid_folder = r'D:\seoul\grids'
# 输出文件夹路径
output_folder = r'D:\seoul\b_satellite_img\shapefiles_clipped_with_fid'
# 遍历所有 grid shapefile
for grid_file in os.listdir(grid_folder):
    if grid_file.endswith('.shp'):
        grid_path = os.path.join(grid_folder, grid_file)

        for input_shp in shapefile_files:
            input_name = os.path.splitext(os.path.basename(input_shp))[0]
            grid_name = os.path.splitext(os.path.basename(grid_file))[0]
            output_name = f'{input_name}_clipped_by_{grid_name}_with_fid.shp'
            output_shp = os.path.join(output_folder, output_name)
            if os.path.exists(output_shp):
                print(f"⏩ 已存在，跳过: {output_shp}")
                continue
            fast_clip_and_assign_fid(input_shp, grid_path, output_shp)

📂 加载输入数据：D:\seoul\b_satellite_img\shapefiles\clipped_20160924_B10_cleaned.shp
✂️ 正在执行 overlay 精准裁剪...
✅ 已保存: D:\seoul\b_satellite_img\shapefiles_clipped_with_fid\clipped_20160924_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
📂 加载输入数据：D:\seoul\b_satellite_img\shapefiles\clipped_20230616_B10_cleaned.shp
✂️ 正在执行 overlay 精准裁剪...
✅ 已保存: D:\seoul\b_satellite_img\shapefiles_clipped_with_fid\clipped_20230616_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
📂 加载输入数据：D:\seoul\b_satellite_img\shapefiles\clipped_20160807_B10_cleaned.shp
✂️ 正在执行 overlay 精准裁剪...
✅ 已保存: D:\seoul\b_satellite_img\shapefiles_clipped_with_fid\clipped_20160807_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
📂 加载输入数据：D:\seoul\b_satellite_img\shapefiles\clipped_20230819_B10_cleaned.shp
✂️ 正在执行 overlay 精准裁剪...
✅ 已保存: D:\seoul\b_satellite_img\shapefiles_clipped_with_fid\clipped_20230819_B10_cleaned_clipped_by_grid_1080m_with_fid.shp
📂 加载输入数据：D:\seoul\b_satellite_img\shapefiles\clipped_20160924_B10_cleaned.shp
✂️ 正在执行 overlay 精准

# 先将数据洗成摄氏度 bt -> lst

In [58]:
import rasterio
import numpy as np
import os


def calculate_lst(input_files, output_dir, multiplier=0.00341802, add_constant=149.0, nodata_value=-9999):
    """
    Calculate Land Surface Temperature (LST) from a list of raster files and save the output.

    Args:
        input_files (list): List of paths to input raster files.
        output_dir (str): Directory to save the output raster files.
        multiplier (float): Multiplication factor for converting to temperature.
        add_constant (float): Additive constant for converting to temperature.
        nodata_value (float): Value to use for no-data in the output files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for input_file in input_files:
        with rasterio.open(input_file) as src:
            # Read the first band and convert to float
            band_data = src.read(1).astype(float)
            profile = src.profile

        # Replace values <= 0 with NaN
        band_data[band_data <= 0] = np.nan

        # Calculate brightness temperature (BT)
        lst_kelvin = (band_data * multiplier) + add_constant

        # Convert to Celsius
        lst_celsius = lst_kelvin - 273.15

        # Replace NaN values with the nodata value
        lst_celsius[np.isnan(lst_celsius)] = nodata_value

        # Update profile for the output file
        profile.update({
            'dtype': 'float32',
            'count': 1,
            'nodata': nodata_value
        })

        # Generate a short output file name
        base_name = os.path.basename(input_file).split('.')[0]
        output_file = os.path.join(output_dir, f"{base_name}_LST.tif")

        # Save the LST raster
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(lst_celsius, 1)

        print(f"LST saved to: {output_file}")

extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files = [
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\\normal\clipped\clean\clipped_{normal_heat[1]}_B10_cleaned.tif'
]
extreme_files = [
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[0]}_B10_cleaned.tif',
    rf'D:\seoul\b_satellite_img\extreme\clipped\clean\clipped_{extreme_heat[1]}_B10_cleaned.tif'
]

# 输出目录
output_dir_normal = r'D:\seoul\b_satellite_img\\normal_heat_day_lst'
output_dir_extreme = r'D:\seoul\b_satellite_img\\extreme_heat_day_lst'

# 计算并保存 LST
calculate_lst(normal_files, output_dir_normal)
calculate_lst(extreme_files, output_dir_extreme)

LST saved to: D:\seoul\b_satellite_img\\normal_heat_day_lst\clipped_20160519_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\normal_heat_day_lst\clipped_20230616_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\extreme_heat_day_lst\clipped_20160807_B10_cleaned_LST.tif
LST saved to: D:\seoul\b_satellite_img\\extreme_heat_day_lst\clipped_20230819_B10_cleaned_LST.tif


# 相减，然后得到HR map

In [60]:
import rasterio
import numpy as np
import os

def calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999):
    os.makedirs(output_dir, exist_ok=True)

    if len(normal_files) != len(extreme_files):
        raise ValueError("Number of normal and extreme files must be the same.")

    for normal_file, extreme_file in zip(normal_files, extreme_files):
        with rasterio.open(normal_file) as normal_src, rasterio.open(extreme_file) as extreme_src:
            normal_data = normal_src.read(1).astype(float)
            extreme_data = extreme_src.read(1).astype(float)

            if normal_data.shape != extreme_data.shape:
                raise ValueError(f"Dimension mismatch between {normal_file} and {extreme_file}.")

            # 强制处理 nodata
            if normal_src.nodata is not None:
                normal_data[normal_data == normal_src.nodata] = np.nan
            else:
                normal_data[normal_data == nodata_value] = np.nan

            if extreme_src.nodata is not None:
                extreme_data[extreme_data == extreme_src.nodata] = np.nan
            else:
                extreme_data[extreme_data == nodata_value] = np.nan

            # 差值
            heat_resilience = normal_data - extreme_data

            # 打印调试
            print(f"\n📁 Processing: {os.path.basename(normal_file)} vs {os.path.basename(extreme_file)}")
            print("🔹 First 10 values from normal_data (non-NaN):", normal_data[~np.isnan(normal_data)].flatten()[:10])
            print("🔹 First 10 values from extreme_data (non-NaN):", extreme_data[~np.isnan(extreme_data)].flatten()[:10])
            print("🔸 First 10 values from heat_resilience (non-NaN):", heat_resilience[~np.isnan(heat_resilience)].flatten()[:10])

            # 替换 NaN 为 nodata
            heat_resilience[np.isnan(heat_resilience)] = nodata_value

            profile = normal_src.profile
            profile.update({
                'dtype': 'float32',
                'nodata': nodata_value
            })

            year = os.path.basename(normal_file).split('_')[1][:4]
            output_file = os.path.join(output_dir, f"{year}_heat_resilience.tif")

            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heat_resilience.astype('float32'), 1)

            print(f"✅ Heat resilience raster saved to: {output_file}")


extreme_heat=[20160807, 20230819]
normal_heat=[20160924, 20230616]
# 输入的栅格文件路径
normal_files_lst = [
    rf'D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_{normal_heat[0]}_B10_cleaned_lst.tif',
    rf'D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_{normal_heat[1]}_B10_cleaned_lst.tif'
]
extreme_files_lst = [
    rf'D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_{extreme_heat[0]}_B10_cleaned_lst.tif',
    rf'D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_{extreme_heat[1]}_B10_cleaned_lst.tif'
]

# 输出目录
output_dir = r'D:\seoul\b_satellite_img\heat_resilience'

# 计算并保存 heat resilience
calculate_heat_resilience(normal_files_lst, extreme_files_lst, output_dir, nodata_value=-9999)


📁 Processing: clipped_20160519_B10_cleaned_lst.tif vs clipped_20160807_B10_cleaned_lst.tif
🔹 First 10 values from normal_data (non-NaN): [27.67844772 28.25609398 27.36399078 27.54172707 27.91770935 28.48168373
 27.86302185 27.99632454 28.20482254 28.24584007]
🔹 First 10 values from extreme_data (non-NaN): [32.88751221 32.70977402 32.58330536 32.68584824 35.6390152  36.1790657
 36.73620224 34.06672668 34.63412094 35.2767067 ]
🔸 First 10 values from heat_resilience (non-NaN): [-1.92776489 -1.97219849 -2.11233521 -2.22854996 -1.54836273 -1.54836273
 -1.33302689 -1.35695267 -1.94827271 -1.96536255]
✅ Heat resilience raster saved to: D:\seoul\b_satellite_img\heat_resilience\2016_heat_resilience.tif

📁 Processing: clipped_20230616_B10_cleaned_lst.tif vs clipped_20230819_B10_cleaned_lst.tif
🔹 First 10 values from normal_data (non-NaN): [26.51290321 27.26828575 26.66671371 27.00509834 26.90255737 27.70921135
 27.27512169 27.61008835 27.86302185 27.91087341]
🔹 First 10 values from extreme_data

# 计算面积和占比

In [ ]:
import geopandas as gpd
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
def add_lst_ratio_within_only(clipped_gdf, city_path, output_path):
    # ✅ STEP 1: 读取 grid 文件（即 city_gdf），用其 CRS 作为标准
    city_gdf = gpd.read_file(city_path)
    target_crs = city_gdf.crs  # ⬅️ 使用它为主
    clipped = clipped_gdf.to_crs(target_crs)
    city_gdf = city_gdf.to_crs(target_crs)

    # ✅ STEP 2: 创建 city_id 和 Shape_Area
    city_gdf = city_gdf.reset_index(drop=True)
    city_gdf['city_id'] = city_gdf.index
    # city_gdf['Shape_Area'] = city_gdf.geometry.area
    city_gdf['lst'] = 0.0  # 初始化

    # ✅ STEP 3: 找出每个 grid 内完全包含的 clipped polygon
    for idx, row in city_gdf.iterrows():
        grid_geom = row.geometry
        inside = clipped[clipped.within(grid_geom)]
        if not inside.empty:
            area_sum = inside.geometry.area.sum()
            city_gdf.at[idx, 'lst'] = area_sum

    # ✅ STEP 4: 计算 lst_ratio
    city_gdf['lst_ratio'] = city_gdf['lst'] / city_gdf['Shape_Area']

    # ✅ STEP 5: 导出为 ITRF_2000_UTM_K
    itrf_proj = (
        '+proj=tmerc +lat_0=38 +lon_0=127.5 '
        '+k=0.9996 +x_0=1000000 +y_0=2000000 '
        '+ellps=GRS80 +units=m +no_defs'
    )
    city_gdf_final = city_gdf.to_crs(itrf_proj)
    city_gdf_final.to_file(output_path, driver='ESRI Shapefile')

    print(f"✅ 完全包含统计完成并保存: {output_path}")




# === 设置路径 ===
# 所有 clipped shapefiles（一个 per grid）
clipped_folder = r'D:\seoul\b_satellite_img\shapefiles_clipped_with_fid'
clipped_files = [f for f in os.listdir(clipped_folder) if f.endswith('.shp')]

# 所有 grid shapefiles
grid_folder = r'D:\seoul\grids'
grid_files = [f for f in os.listdir(grid_folder) if f.endswith('.shp')]

# 输出文件夹
output_folder = r'D:\seoul\grids\city_ratio_outputs'
os.makedirs(output_folder, exist_ok=True)

def process_pair(grid_file):
    try:
        grid_name = os.path.splitext(grid_file)[0]
        grid_path = os.path.join(grid_folder, grid_file)

        matched_clipped = [
            f for f in clipped_files if grid_name in f
        ]

        if not matched_clipped:
            print(f"⚠️ 找不到与 {grid_file} 匹配的 clipped 文件")
            return

        clipped_path = os.path.join(clipped_folder, matched_clipped[0])
        clipped_gdf = gpd.read_file(clipped_path)
        clipped_gdf = clipped_gdf.reset_index(drop=True)
        clipped_gdf['city_id'] = clipped_gdf['FID'] if 'FID' in clipped_gdf.columns else clipped_gdf.index

        output_path = os.path.join(output_folder, f'city2020_lst_ratio_{grid_name}.shp')
        if os.path.exists(output_path):
            print(f"⏩ 已存在，跳过: {output_path}")
            return

        add_lst_ratio_within_only(clipped_gdf, grid_path, output_path)

    except Exception as e:
        print(f"❌ 处理 {grid_file} 失败: {e}")

# 🚀 并行处理
print("🚀 开始批量匹配 grid ↔ clipped...")
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(process_pair, grid_files)

In [ ]:
import geopandas as gpd

file = r'D:\seoul\Final_data\building\Raw_data\N1A_B0010000.shp'
data = gpd.read_file(file)
data


In [ ]:
print(data["종류"].unique())
print(data["종류"].value_counts())

In [ ]:
# 무벽건물
# 가건물

In [1]:
import geopandas as gpd

file = r'D:\seoul\c_data\a_building\\AL_11_D010_20161105\AL_11_D010_20161105.shp'

data_small = gpd.read_file(file, encoding='euc_kr')#, rows=50000
data_small

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A14,A15,A16,A17,A18,A19,A20,A21,A22,geometry
0,24937,1988197105154543659600000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,4686.15,0.0,0.0,0.0,0.0,2381,0,B00100000000TT1VW,1899-12-30,"POLYGON ((197119.187 454372.508, 197103.397 45..."
1,24922,1988197172864542613800000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,6572.10,0.0,0.0,0.0,0.0,2382,0,B00100000000TT1GH,1899-12-30,"POLYGON ((197167.005 454248.176, 197144.745 45..."
2,24935,1988197056224543775000000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,5489.43,0.0,0.0,0.0,0.0,2383,0,B00100000000TT1TU,1899-12-30,"POLYGON ((197077.797 454396.579, 197045.366 45..."
3,24936,1988197076494543686200000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,4772.55,0.0,0.0,0.0,0.0,2384,0,B00100000000TT1UV,1899-12-30,"POLYGON ((197092.447 454380.448, 197070.876 45..."
4,24958,1988197149274543345500000000,1111010100100010000,1111010100,서울특별시 종로구 청운동,1,1,일반,02000,공동주택,...,6732.21,0.0,0.0,0.0,0.0,2385,0,B00100000000TT2GI,1899-12-30,"POLYGON ((197183.247 454318.456, 197174.476 45..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746183,5613,0000215570844507756700000000,1174011000200080011,1174011000,서울특별시 강동구 강일동,8-11,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WP4CF,1899-12-30,"POLYGON ((215570.446 450772.851, 215569.000 45..."
746184,5523,0000215740974502818900000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WOHET,1899-12-30,"POLYGON ((215737.971 450288.975, 215749.246 45..."
746185,5518,0000215654324502800000000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WOHFU,1899-12-30,"POLYGON ((215649.260 450282.765, 215659.061 45..."
746186,5519,0000215654644502750400000000,1174011000200240001,1174011000,서울특별시 강동구 강일동,24-1,2,산,None,None,...,0.00,0.0,0.0,0.0,0.0,None,None,B00100000000WPCKV,1899-12-30,"POLYGON ((215659.612 450273.575, 215649.804 45..."


In [2]:
print(data_small["A8"].unique())

['02000' None '05000' '01000' '11000' '06000' '03000' '04000' '13000'
 '10000' 'Z8000' 'Z5000' '20000' '14000' 'Z3000' '17000' '19000' '18000'
 '15000' '09000' '27000' 'Z9000' '16000' '07000' '12000' '21000' '08000'
 'Z6000' '24000' '22000' '23000' '25000' '26000' '29000' '창고']
['공동주택' None '문화및집회시설' '단독주택' '노유자시설' '종교시설' '제1종근린생활시설' '제2종근린생활시설'
 '운동시설' '교육연구시설' '교육연구및복지시설' '자동차관련시설' '업무시설' '근린생활시설' '공장' '위험물저장및처리시설'
 '창고시설' '숙박시설' '의료시설' '관광휴게시설' '공공용시설' '위락시설' '판매시설' '수련시설' '동.식물 관련시설'
 '운수시설' '판매및영업시설' '방송통신시설' '분뇨.쓰레기처리시설' '교정및군사시설' '발전시설' '묘지관련시설' '장례식장']


In [4]:
import geopandas as gpd

file = r'D:\seoul\c_data\a_building\\AL_D010_11_20240607.shp'

data_small = gpd.read_file(file, encoding='euc_kr')#, rows=50000
data_small

,A0,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,A20,A21,A22,A23,A24,A25,A26,A27,A28,geometry
0,32,1991201839054527769900000000,1111017500107040000,1111017500,서울특별시 종로구 숭인동,704,1,일반,01000,단독주택,...,N,B00100000000T30Z9,2024-06-04,11110,None,None,3.0,0.0,2017-05-30,"POLYGON ((201913.638 553085.323, 201912.228 55..."
1,34,1967201343844527773800000000,1111017500100560024,1111017500,서울특별시 종로구 숭인동,56-24,1,일반,01000,단독주택,...,N,B00100000000T311B,2024-06-04,11110,None,None,3.0,0.0,2018-06-19,"POLYGON ((201413.758 553078.463, 201410.198 55..."
2,36,1962201375194527742000000000,1111017500100560054,1111017500,서울특별시 종로구 숭인동,56-54,1,일반,01000,단독주택,...,N,B00100000000T313D,2024-06-04,11110,None,None,2.0,0.0,2017-02-24,"POLYGON ((201442.598 553075.703, 201442.298 55..."
3,37,1979201388484527710100000000,1111017500100560053,1111017500,서울특별시 종로구 숭인동,56-53,1,일반,01000,단독주택,...,N,B00100000000T314E,2024-06-04,11110,None,None,2.0,1.0,2018-11-06,"POLYGON ((201458.568 553069.403, 201453.548 55..."
4,38,0000200468294527764000000000,1111016500100280011,1111016500,서울특별시 종로구 이화동,28-11,1,일반,None,None,...,None,B00100000000T315F,2024-06-04,11110,None,None,0.0,0.0,None,"POLYGON ((200540.538 553080.543, 200536.438 55..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696062,10105,1990198254284494774700000000,1117010100102670002,1117010100,서울특별시 용산구 후암동,267-2,1,일반,01000,단독주택,...,Y,B001000000013AU3O,2024-06-04,11170,None,None,2.0,1.0,2024-05-23,"POLYGON ((198316.762 549783.247, 198323.391 54..."
696063,8522,1995198146964497771000000000,1117010100100480012,1117010100,서울특별시 용산구 후암동,48-12,1,일반,01000,단독주택,...,Y,B00100000000RMMAN,2024-06-04,11170,None,None,2.0,1.0,2024-05-23,"POLYGON ((198212.062 550079.067, 198212.682 55..."
696064,12847,2003201075314427046800000000,1165010800115930007,1165010800,서울특별시 서초구 서초동,1593-7,1,일반,02000,공동주택,...,N,B001000000012Z3TC,2024-06-04,11650,서초 이오빌,None,24.0,6.0,2024-05-23,"POLYGON ((201133.460 543032.394, 201136.678 54..."
696065,9841,1993205799614483901100000000,1121510500108510019,1121510500,서울특별시 광진구 자양동,851-19,1,일반,03000,제1종근린생활시설,...,N,B00100000000W3WOX,2024-06-04,11215,None,None,5.0,1.0,2024-05-23,"POLYGON ((205871.117 548687.488, 205862.148 54..."


In [ ]:
print(data_small["A8"].unique())
print(data_small["A9"].unique())